# 04 — Inspect retrieval misses

When a query scored Recall@5 = 0 (or worse than expected), pull the actual top-K chunks and the expected document's chunks side-by-side. Helps answer: was the right doc *almost* retrieved? Which distractors won? Are the misses concentrated on one source?

Re-runs retrieval directly against a Chroma collection that was built by `eval-retrieval` (collections persist post-run unless explicitly deleted).

In [ ]:
from pathlib import Path

import chromadb
import pandas as pd

from evals.retrieval.cache import CachedEmbedder
from evals.retrieval.dataset import load_eval_set
from retrievers.embedding import OpenAIEmbedder

EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIMS = 1536
EVAL_SET = Path("datasets/retrieval_eval.jsonl")
CACHE_DIR = Path("data/eval_results/.embedding_cache")

In [ ]:
# Connect to the local Chroma container the eval ran against.
client = chromadb.HttpClient(host="localhost", port=8000)
collections = [c.name for c in client.list_collections()]
[c for c in collections if c.startswith("eval_")]

In [ ]:
# Pick a specific eval-run collection and the source it indexes.
COLLECTION = "eval_<timestamp>_raw_store"  # <-- fill in
SOURCE = "raw_store"
coll = client.get_collection(COLLECTION)
embedder = CachedEmbedder(
    OpenAIEmbedder(EMBEDDING_MODEL, EMBEDDING_DIMS), cache_dir=CACHE_DIR
)

In [ ]:
# Load eval pairs for this source and walk through them, scoring + flagging misses.
pairs = [p for p in load_eval_set(EVAL_SET) if p.source == SOURCE]
rows = []
for p in pairs:
    [vec] = embedder.embed_batch([p.query])
    resp = coll.query(
        query_embeddings=[vec], n_results=10, include=["metadatas", "documents"]
    )
    retrieved_ids = [m["content_id"] for m in resp["metadatas"][0]]
    rows.append(
        {
            "query": p.query,
            "expected": p.expected_content_id,
            "hit@5": int(p.expected_content_id in retrieved_ids[:5]),
            "top_5": retrieved_ids[:5],
        }
    )
df = pd.DataFrame(rows)
df

In [ ]:
# Drill into one miss — show the actual chunks retrieved alongside the expected doc's chunks.
MISS = df[df["hit@5"] == 0].iloc[0]
print(f"Query:    {MISS['query']}")
print(f"Expected: {MISS['expected']}\n")

[vec] = embedder.embed_batch([MISS["query"]])
got = coll.query(query_embeddings=[vec], n_results=5, include=["metadatas", "documents"])
want = coll.get(
    where={"content_id": MISS["expected"]}, include=["metadatas", "documents"]
)

print("--- top-5 retrieved ---")
for m, d in zip(got["metadatas"][0], got["documents"][0]):
    print(f"  [{m['content_id']}::{m['chunk_index']}]  {d[:120]}…")

print("\n--- expected document's chunks ---")
for m, d in zip(want["metadatas"], want["documents"]):
    print(f"  [{m['content_id']}::{m['chunk_index']}]  {d[:120]}…")